In [0]:
from pyspark.sql.functions import hash, abs

In [0]:
dbutils.widgets.text("catalog_name", "dbr_dev")
dbutils.widgets.text("gold_schema_name", "weather_gold")
dbutils.widgets.text("governance_schema_name", "weather_governance")

catalog_name = dbutils.widgets.get("catalog_name")
gold_schema_name = dbutils.widgets.get("gold_schema_name")
governance_schema_name = dbutils.widgets.get("governance_schema_name")

### RLS

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog_name}.{governance_schema_name}.specialists
(    
    city_id STRING,
    user STRING
) 
USING DELTA
LOCATION 'abfss://dataweather@dlspl21databricks.dfs.core.windows.net/governance/tables/specialists';
""")



In [0]:
current_user = spark.sql("SELECT current_user()").collect()[0][0]

In [0]:
spark.sql(f"""
INSERT INTO {catalog_name}.{governance_schema_name}.specialists VALUES
(    
    abs(hash("Warszawa")),
    current_user
)
""")



In [0]:
spark.sql(f"""          
CREATE OR REPLACE FUNCTION row_filter(city_idx STRING)
RETURN IS_ACCOUNT_GROUP_MEMBER('weather_admins') OR EXISTS
(
    SELECT 1 FROM {catalog_name}.{governance_schema_name}.specialists s  
    WHERE s.city_id = city_idx AND s.user = CURRENT_USER
)         
""")
   

In [0]:
spark.sql(f"""          
ALTER TABLE {catalog_name}.{gold_schema_name}.fact_rainfall
SET ROW FILTER row_filter ON (city_id);          
""")   


spark.sql(f"""          
ALTER TABLE {catalog_name}.{gold_schema_name}.fact_weather
SET ROW FILTER row_filter ON (city_id);          
""")   